# 1. Fundamentos de Agentes de IA

## Objetivos de Aprendizaje
- Comprender qué es un agente de IA y cuáles son sus componentes clave.
- Entender el ciclo de razonamiento (ReAct) que sigue un agente para resolver problemas.
- Implementar un agente simple desde cero en Python (Vanilla Code).
- Reconocer las limitaciones de un agente básico y la necesidad de frameworks como LangChain o CrewAI.

## ¿Qué es un Agente de IA?

Un agente de IA no es simplemente un modelo de lenguaje (LLM) que responde a preguntas. Es un sistema más avanzado que utiliza un LLM como su **cerebro (core engine)** para razonar y tomar decisiones. A diferencia de una simple llamada a una API, un agente puede:

1.  **Descomponer un objetivo complejo** en una secuencia de pasos intermedios.
2.  **Interactuar con herramientas externas** (APIs, bases de datos, funciones de código) para obtener información o ejecutar acciones en el mundo real.
3.  **Observar los resultados** de esas acciones y ajustar su plan en consecuencia.
4.  **Repetir este ciclo** hasta que el objetivo original se haya cumplido.

Piénsalo como un becario inteligente: le das una tarea de alto nivel (ej. "Investiga el precio de las acciones de Apple y dime si es un buen momento para comprar"), y él solo descubre qué herramientas usar (búsqueda web, una API financiera), cómo usarlas y cómo interpretar los resultados para darte una recomendación.

### Componentes Clave de un Agente

Un agente, en su forma más básica, se compone de tres elementos principales:

1.  **Cerebro (Core Engine)**: El LLM que impulsa al agente. Es responsable del razonamiento, la planificación y la toma de decisiones.
2.  **Memoria (Memory)**: Un sistema para almacenar y recuperar información de la conversación actual (memoria a corto plazo) o de interacciones pasadas (memoria a largo plazo). Esto le da contexto al agente.
3.  **Herramientas (Tools)**: Funciones o APIs que el agente puede "llamar" para interactuar con el mundo exterior. Esto supera la limitación del conocimiento estático del LLM.

## Implementación de un Agente Básico en Python

Los notebooks usan la API de OpenAI directamente. Necesitas una API key de OpenAI guardada en el archivo `.env` como `OPENAI_API_KEY`.

Link: https://platform.openai.com/api-keys

In [1]:
import os
import re
import json
from openai import OpenAI
from dotenv import load_dotenv
from datetime import datetime

# --- 1. Configuración del Cliente OpenAI ---
# Asegúrate de tener la variable de entorno OPENAI_API_KEY configurada en el archivo .env
load_dotenv()

try:
    client = OpenAI(
        api_key=os.environ.get("OPENAI_API_KEY")
    )
    print("✅ Cliente OpenAI configurado correctamente.")
except Exception as e:
    print(f"❌ Error configurando el cliente: {e}")
    client = None

✅ Cliente OpenAI configurado correctamente.


### 2. Definición de las Herramientas (Tools)

Vamos a crear dos herramientas muy simples que nuestro agente podrá usar:

In [2]:
def get_current_time(args):
    """Devuelve la fecha y hora actual."""
    return f"La fecha y hora actual es: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

def search_web(args):
    """Simula una búsqueda web para un término dado."""
    query = args.get("query", "")
    # En una implementación real, aquí llamaríamos a una API de búsqueda (ej. Google, Bing)
    print(f"🔎 Buscando en la web: '{query}'...")
    if "elon musk" in query.lower():
        return "Elon Musk es el CEO de SpaceX y Tesla."
    elif "inteligencia artificial" in query.lower():
        return "La IA es un campo de la informática dedicado a crear sistemas que pueden realizar tareas que normalmente requieren inteligencia humana."
    else:
        return f"No se encontraron resultados para '{query}'."
    # api provider


# Mapeo de herramientas para que el agente sepa qué funciones puede llamar
tools = {
    "get _current_time": {
        "function": get_current_time,
        "description": "Útil para obtener la fecha y hora actual.",
        "args": {}
    },
    "search_web": {
        "function": search_web,
        "description": "Útil para buscar información en internet sobre personas, lugares o conceptos.",
        "args": {"query": "la pregunta a buscar"}
    }
}

print("✅ Herramientas del agente definidas.")

✅ Herramientas del agente definidas.


In [3]:
tool_descs = "\n".join(
        f"- {name}: {details['description']} Argumentos: {json.dumps(details['args'], ensure_ascii=False)}"
        for name, details in tools.items()
    )



In [5]:

tool_descs

'- get _current_time: Útil para obtener la fecha y hora actual. Argumentos: {}\n- search_web: Útil para buscar información en internet sobre personas, lugares o conceptos. Argumentos: {"query": "la pregunta a buscar"}'

### 3. El Cerebro del Agente y el Ciclo ReAct

Ahora, la parte más importante: el **ciclo de razonamiento**. Usaremos un enfoque llamado **ReAct (Reason + Act)**. En cada paso, el LLM decide una de estas tres cosas:

1.  **Reason (Razonar)**: Piensa cuál es el siguiente paso lógico para alcanzar el objetivo.
2.  **Act (Actuar)**: Elige y utiliza una de las herramientas disponibles.
3.  **Answer (Responder)**: Si ya tiene suficiente información, da la respuesta final al usuario.

Para guiar al LLM, usaremos un **prompt de sistema** muy específico que le enseñe este patrón de pensamiento.

In [6]:
def create_system_prompt(tools):
    tool_descs = "\n".join(
        f"- {name}: {details['description']} Argumentos: {json.dumps(details['args'], ensure_ascii=False)}"
        for name, details in tools.items()
    )

    return f"""Eres un asistente útil que puede usar herramientas para responder preguntas.
Responde SIEMPRE en texto plano (sin markdown ni negritas) siguiendo estrictamente este formato:

Thought: tu razonamiento sobre qué hacer a continuación.
Action: {{"tool": "nombre_de_la_herramienta", "args": {{"nombre_argumento": "valor"}}}}

Después de escribir la línea Action detente y espera la Observation.
Cuando ya tengas toda la información, responde en su lugar con:

Thought: ya tengo toda la información necesaria.
Final Answer: la respuesta final a la pregunta original.

Reglas:
- Action debe ser un único objeto JSON válido con las claves "tool" y "args".
- "tool" debe ser una de: {json.dumps(list(tools.keys()))}
- Usa solo una herramienta por turno.
- Nunca escribas ni inventes tú mismo la Observation.

Herramientas disponibles:
{tool_descs}"""


def _extract_json_object(text, start):
    """Extrae el objeto JSON completo que empieza en text[start], contando llaves."""
    depth, in_string, escaped = 0, False, False
    for i in range(start, len(text)):
        ch = text[i]
        if in_string:
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == '"':
                in_string = False
        elif ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None


# Toleran "Action:", "**Action (Acción):**", "Action (Accion) :", etc.
ACTION_RE = re.compile(r"Action\s*(?:\([^)]*\))?\s*:\**\s*", re.IGNORECASE)
FINAL_RE = re.compile(r"Final\s*Answer\s*(?:\([^)]*\))?\s*:\**\s*", re.IGNORECASE)


def parse_action(text):
    """Devuelve el dict de la acción, o None si el texto no contiene una."""
    match = ACTION_RE.search(text)
    if not match:
        return None
    start = text.find("{", match.end())
    if start == -1:
        return None
    raw = _extract_json_object(text, start)
    if raw is None:
        return None
    return json.loads(raw)


def run_agent(user_query, client, tools, max_steps=5):
    if not client:
        print("❌ Cliente no inicializado.")
        return

    messages = [
        {"role": "system", "content": create_system_prompt(tools)},
        {"role": "user", "content": user_query},
    ]

    print(f"--- Agente iniciado para la consulta: '{user_query}' ---")

    for step in range(max_steps):
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            temperature=0,
            max_tokens=500,
            stop=["Observation:"],  # evita que el modelo invente la observación
        )

        text = (response.choices[0].message.content or "").strip()
        messages.append({"role": "assistant", "content": text})
        print(f"\n🤖 Paso {step + 1} - Pensamiento del Agente:\n{text}")

        final = FINAL_RE.search(text)
        if final:
            print("\n--- ✅ Agente ha finalizado ---")
            return text[final.end():].strip()

        error = None
        try:
            action = parse_action(text)
        except json.JSONDecodeError as e:
            action = None
            error = f"El JSON de la acción no es válido ({e}). Usa: Action: {{\"tool\": ..., \"args\": {{...}}}}"

        if action is None:
            observation = error or (
                'No encontré una acción válida. Responde con '
                '\'Action: {"tool": ..., "args": {...}}\' o con \'Final Answer: ...\'.'
            )
        else:
            tool_name = action.get("tool")
            tool_args = action.get("args") or {}
            if isinstance(tool_args, str):
                tool_args = {"query": tool_args}

            if tool_name in tools:
                try:
                    observation = tools[tool_name]["function"](tool_args)
                except Exception as e:
                    observation = f"Error al ejecutar '{tool_name}': {e}"
            else:
                observation = (
                    f"Herramienta '{tool_name}' desconocida. "
                    f"Disponibles: {json.dumps(list(tools.keys()))}"
                )

        print(f"👀 Observation: {observation}")
        messages.append({"role": "user", "content": f"Observation: {observation}"})

    print("\n--- 🛑 Límite de iteraciones alcanzado. ---")
    return "El agente no pudo completar la tarea en el número máximo de pasos."


print("✅ Lógica del agente definida.")


✅ Lógica del agente definida.


### 4. Ejecución del Agente

Ahora, pongamos a nuestro agente a trabajar con una pregunta que requiere usar una herramienta.

In [7]:
import json
final_response = run_agent("dame informacion sobre Elon Musk y qué hora es?", client, tools)
print(f"🏁 Respuesta Final del Agente: {final_response}")

--- Agente iniciado para la consulta: 'dame informacion sobre Elon Musk y qué hora es?' ---

🤖 Paso 1 - Pensamiento del Agente:
Thought: Necesito obtener la hora actual primero y luego buscar información sobre Elon Musk.
Action: {"tool": "get _current_time", "args": {}}
👀 Observation: La fecha y hora actual es: 2026-09-10 17:06:48

🤖 Paso 2 - Pensamiento del Agente:
Thought: Ahora que tengo la hora actual, buscaré información sobre Elon Musk.
Action: {"tool": "search_web", "args": {"query": "Elon Musk información 2026"}}
🔎 Buscando en la web: 'Elon Musk información 2026'...
👀 Observation: Elon Musk es el CEO de SpaceX y Tesla.

🤖 Paso 3 - Pensamiento del Agente:
Thought: ya tengo toda la información necesaria.
Final Answer: La fecha y hora actual es 10 de septiembre de 2026 a las 17:06:48. Elon Musk es el CEO de SpaceX y Tesla.

--- ✅ Agente ha finalizado ---
🏁 Respuesta Final del Agente: La fecha y hora actual es 10 de septiembre de 2026 a las 17:06:48. Elon Musk es el CEO de SpaceX y

In [8]:
final_response


'La fecha y hora actual es 10 de septiembre de 2026 a las 17:06:48. Elon Musk es el CEO de SpaceX y Tesla.'

## Conclusiones y Próximos Pasos

Hemos construido un agente funcional desde cero. Sin embargo, hemos tenido que manejar mucha lógica compleja:

- **Análisis de la respuesta del LLM**: Usar expresiones regulares (`re`) y `json.loads` para extraer la acción es frágil y propenso a errores.
- **Gestión del prompt**: Construir y actualizar el prompt manualmente es tedioso.
- **Manejo del ciclo**: El bucle `for` con la lógica de parada es repetitivo.
- **Escalabilidad**: Añadir más herramientas, gestionar la memoria o implementar planes más complejos se volvería muy difícil.

**Aquí es donde entran los frameworks como LangChain y CrewAI.**

Estos frameworks abstraen toda esta complejidad, permitiéndonos definir agentes, herramientas y tareas de una manera mucho más declarativa y robusta. En los próximos notebooks, veremos cómo recrear este mismo agente usando estas herramientas para apreciar la diferencia en simplicidad y potencia.